# Phase 05 — Statistical Validation

This notebook statistically validates selected relationships identified
during exploratory data analysis (EDA).

Validation includes:
- Statistical significance
- Effect size
- Confidence intervals where appropriate
- Analytical and HR/business interpretation

In [13]:
import pandas as pd
import numpy as np

from scipy import stats

import warnings

warnings.filterwarnings("ignore")

print("Statistical validation environment loaded successfully.")

Statistical validation environment loaded successfully.


In [14]:
DATA_PATH = "../data/processed/retainx_hr_1000.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Number of employees: {len(df)}")

Dataset shape: (1000, 26)
Number of employees: 1000


In [15]:
print("Attrition distribution:")
print(df["Attrition"].value_counts())

print("\nAttrition percentage:")
print(df["Attrition"].value_counts(normalize=True).mul(100).round(2))

Attrition distribution:
Attrition
No     788
Yes    212
Name: count, dtype: int64

Attrition percentage:
Attrition
No     78.8
Yes    21.2
Name: proportion, dtype: float64


In [16]:
# Variables required for Phase 05
required_columns = [
    "Attrition",
    "Job_Satisfaction",
    "Department",
    "Monthly_Income",
    "Years_Since_Last_Promotion"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")

All required columns are present.


In [17]:
print("Dataset columns:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Dataset columns:
1. Employee_ID
2. Age
3. Gender
4. Marital_Status
5. Department
6. Job_Role
7. Job_Level
8. Monthly_Income
9. Hourly_Rate
10. Years_at_Company
11. Years_in_Current_Role
12. Years_Since_Last_Promotion
13. Work_Life_Balance
14. Job_Satisfaction
15. Performance_Rating
16. Training_Hours_Last_Year
17. Overtime
18. Project_Count
19. Average_Hours_Worked_Per_Week
20. Absenteeism
21. Work_Environment_Satisfaction
22. Relationship_with_Manager
23. Job_Involvement
24. Distance_From_Home
25. Number_of_Companies_Worked
26. Attrition


In [18]:
print("\nMissing values:")
print(df[required_columns].isnull().sum())


Missing values:
Attrition                     0
Job_Satisfaction              0
Department                    0
Monthly_Income                0
Years_Since_Last_Promotion    0
dtype: int64


In [19]:
print("\nAttrition values:")
print(df["Attrition"].unique())

print("\nJob Satisfaction values:")
print(sorted(df["Job_Satisfaction"].dropna().unique()))

print("\nDepartments:")
print(df["Department"].dropna().unique())


Attrition values:
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

Job Satisfaction values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Departments:
<StringArray>
['Marketing', 'HR', 'IT', 'Sales', 'Finance']
Length: 5, dtype: str


## 4. Test 1 — Job Satisfaction and Attrition

EDA indicated that attrition rates vary across different levels of
job satisfaction.

A Chi-square test of independence is used to determine whether
Job Satisfaction and Attrition are statistically associated.

### Hypotheses

- **H₀ (Null Hypothesis):** Job Satisfaction and Attrition are independent.
- **H₁ (Alternative Hypothesis):** Job Satisfaction and Attrition are associated.

The significance level is set to α = 0.05.

In [26]:
job_satisfaction_table = pd.crosstab(
    df["Job_Satisfaction"],
    df["Attrition"]
)

print("Job Satisfaction vs Attrition:")
print(job_satisfaction_table)

Job Satisfaction vs Attrition:
Attrition          No  Yes
Job_Satisfaction          
1                 132   47
2                 170   51
3                 162   44
4                 156   41
5                 168   29


In [24]:
chi2, p_value, degrees_of_freedom, expected = stats.chi2_contingency(
    job_satisfaction_table
)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {degrees_of_freedom}")
print(f"P-value: {p_value:.6f}")

Chi-square statistic: 8.1775
Degrees of freedom: 4
P-value: 0.085288


In [25]:
n = job_satisfaction_table.to_numpy().sum()
rows, columns = job_satisfaction_table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(rows - 1, columns - 1))
)

print(f"Cramér's V: {cramers_v:.4f}")

Cramér's V: 0.0904


### Interpretation

The Chi-square test produced a statistic of 8.1775 with 4 degrees
of freedom and a p-value of 0.085288.

Since the p-value is greater than the significance level of 0.05,
we fail to reject the null hypothesis.

Therefore, this analysis does not provide sufficient statistical
evidence of an association between Job Satisfaction and Attrition
at the 5% significance level.

Cramér's V is 0.0904, indicating a very weak association between
the two variables.

Although EDA showed differences in attrition rates across job
satisfaction levels, the statistical validation does not support
a strong statistically significant relationship in this dataset.

This result indicates association only and does not imply that
Job Satisfaction causes Attrition.

## 5. Test 2 — Department and Attrition

EDA indicated that attrition rates differ across departments.

A Chi-square test of independence is used to determine whether
Department and Attrition are statistically associated.

### Hypotheses

- **H₀ (Null Hypothesis):** Department and Attrition are independent.
- **H₁ (Alternative Hypothesis):** Department and Attrition are associated.

The significance level is set to α = 0.05.

In [27]:
department_table = pd.crosstab(
    df["Department"],
    df["Attrition"]
)

print("Department vs Attrition:")
print(department_table)

Department vs Attrition:
Attrition    No  Yes
Department          
Finance     159   49
HR          147   35
IT          151   40
Marketing   168   48
Sales       163   40


In [28]:
chi2_department, p_value_department, degrees_of_freedom_department, expected_department = stats.chi2_contingency(
    department_table
)

print(f"Chi-square statistic: {chi2_department:.4f}")
print(f"Degrees of freedom: {degrees_of_freedom_department}")
print(f"P-value: {p_value_department:.6f}")

Chi-square statistic: 1.5291
Degrees of freedom: 4
P-value: 0.821478


In [29]:
n_department = department_table.to_numpy().sum()
rows_department, columns_department = department_table.shape

cramers_v_department = np.sqrt(
    chi2_department /
    (n_department * min(rows_department - 1, columns_department - 1))
)

print(f"Cramér's V: {cramers_v_department:.4f}")

Cramér's V: 0.0391
